# Buchwald-Hartwig Yield Model — v3: Adding Morgan Fingerprints

## Background

v2 (`05_model_v2_fixed_descriptors.ipynb`) fixed a silent RDKit bug and reached **Test R² = 0.72**
using 20 simple physicochemical descriptors per substrate (MW, LogP, TPSA, aromaticity, etc.) plus
categorical base/ligand/additive identity — 71 features total.

Those descriptors are all *bulk* properties: they summarize a molecule with a handful of numbers but
don't encode its actual atom-by-atom structure. Two very different molecules can share similar MW/LogP/TPSA.

## The idea

Add **Morgan fingerprints** (a standard cheminformatics representation: circular substructure fragments
hashed into a fixed-length bit vector) for both the aryl halide and the coupling product. 128 bits,
radius 2, computed with RDKit's (non-deprecated) `rdFingerprintGenerator`. This gives the model direct
access to substructure information the v2 descriptors couldn't see.

## Result

| Version | Features | Test R² (single split) | 5-fold CV R² |
|---|---|---|---|
| v1 (buggy) | 31 (categorical only) | 0.30 | — |
| v2 (fixed descriptors) | 71 (descriptors + categorical) | 0.72 (Gradient Boosting) | 0.741 ± 0.015 |
| v3 (+ Morgan fingerprints) | 327 (71 + 256 fingerprint bits) | **0.93** (XGBoost) | **0.933 ± 0.005** |

The 5-fold CV confirms this isn't a lucky train/test split — both the mean and the very low standard
deviation (0.005) show the improvement is real and stable.


In [1]:
import os, pickle, warnings, time
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

t0 = time.time()
DATA_DIR = "data"
MODELS_DIR = os.path.join(DATA_DIR, "trained_models")
N_BITS = 128

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=N_BITS)

def morgan_bits(smiles):
    try:
        if pd.isna(smiles) or smiles is None or smiles == "":
            return np.zeros(N_BITS, dtype=int)
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            return np.zeros(N_BITS, dtype=int)
        fp = morgan_gen.GetFingerprint(mol)
        arr = np.zeros((N_BITS,), dtype=int)
        for i in fp.GetOnBits():
            arr[i] = 1
        return arr
    except Exception:
        return np.zeros(N_BITS, dtype=int)

X_v2 = pd.read_csv(os.path.join(DATA_DIR, "X_features_scaled_v2.csv"))
y = pd.read_csv(os.path.join(DATA_DIR, "y_target_v2.csv")).values.ravel()
df = pd.read_csv(os.path.join(DATA_DIR, "doyle_buchwald_data_cleaned.csv"))

print("Computing Morgan fingerprints (radius=2, 128 bits) for aryl halide + product...")
aryl_fp = np.vstack([morgan_bits(s) for s in df["aryl_halide_smiles"]])
product_fp = np.vstack([morgan_bits(s) for s in df["product_smiles"]])
aryl_fp_df = pd.DataFrame(aryl_fp, columns=[f"aryl_fp_{i}" for i in range(N_BITS)])
product_fp_df = pd.DataFrame(product_fp, columns=[f"product_fp_{i}" for i in range(N_BITS)])
print(f"  aryl_fp: {aryl_fp_df.shape}, product_fp: {product_fp_df.shape}")

X_v3 = pd.concat([X_v2.reset_index(drop=True), aryl_fp_df, product_fp_df], axis=1)
feature_names_v3 = list(X_v3.columns)
print(f"Combined feature matrix: {X_v3.shape}")

X_v3.to_csv(os.path.join(DATA_DIR, "X_features_scaled_v3.csv"), index=False)
with open(os.path.join(MODELS_DIR, "feature_names_v3.pkl"), "wb") as f:
    pickle.dump(feature_names_v3, f)


Computing Morgan fingerprints (radius=2, 128 bits) for aryl halide + product...
  aryl_fp: (4312, 128), product_fp: (4312, 128)
Combined feature matrix: (4312, 327)


## Model training: v3 vs v2, same train/test split for a fair comparison

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X_v3, y, test_size=0.2, random_state=42)

results = {}
for name, mdl in [
    ("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=42)),
    ("XGBoost", xgb.XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
]:
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test)
    results[name] = {
        "model": mdl,
        "train_r2": r2_score(y_train, mdl.predict(X_train)),
        "test_r2": r2_score(y_test, pred),
        "test_mae": mean_absolute_error(y_test, pred),
        "test_rmse": mean_squared_error(y_test, pred) ** 0.5,
    }

print(f"{'Model':<20}{'Train R2':>12}{'Test R2':>12}{'Test MAE':>12}{'Test RMSE':>12}")
for name, r in results.items():
    print(f"{name:<20}{r['train_r2']:>12.4f}{r['test_r2']:>12.4f}{r['test_mae']:>12.4f}{r['test_rmse']:>12.4f}")

best_name = max(results, key=lambda k: results[k]["test_r2"])
best_model_v3 = results[best_name]["model"]
print(f"\nBest v3 model: {best_name} (Test R2 = {results[best_name]['test_r2']:.4f})")
print(f"v2 baseline (same split, descriptors only): Test R2 = 0.7197 (Gradient Boosting)")
print(f"Delta: {results[best_name]['test_r2'] - 0.7197:+.4f}")


Model                   Train R2     Test R2    Test MAE   Test RMSE
Random Forest             0.9870      0.8938      5.6088      8.9168
Gradient Boosting         0.8194      0.7915      9.4640     12.4929
XGBoost                   0.9978      0.9300      4.8803      7.2373

Best v3 model: XGBoost (Test R2 = 0.9300)
v2 baseline (same split, descriptors only): Test R2 = 0.7197 (Gradient Boosting)
Delta: +0.2103


## Robustness check: 5-fold cross-validation (v3 winner vs v2 baseline)

A single 80/20 split can be lucky or unlucky. Cross-validation gives a more trustworthy number to report.

In [3]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_class = type(best_model_v3)(**best_model_v3.get_params())
cv_scores_v3 = cross_val_score(best_class, X_v3, y, cv=kf, scoring="r2", n_jobs=-1)
print(f"5-fold CV R2 ({best_name}, v3): {cv_scores_v3}")
print(f"  mean = {cv_scores_v3.mean():.4f}, std = {cv_scores_v3.std():.4f}")

gb_v2 = GradientBoostingRegressor(random_state=42)
cv_scores_v2 = cross_val_score(gb_v2, X_v2, y, cv=kf, scoring="r2", n_jobs=-1)
print(f"5-fold CV R2 (Gradient Boosting, v2 baseline): {cv_scores_v2}")
print(f"  mean = {cv_scores_v2.mean():.4f}, std = {cv_scores_v2.std():.4f}")


5-fold CV R2 (XGBoost, v3): [0.93003704 0.93017182 0.9354315  0.94120278 0.92851265]
  mean = 0.9331, std = 0.0047
5-fold CV R2 (Gradient Boosting, v2 baseline): [0.71967628 0.75081366 0.73701713 0.7634967  0.7346865 ]
  mean = 0.7411, std = 0.0149


In [4]:
# Save artifacts: winning model + metrics (Random Forest pickle skipped: not the
# winner here, and its file size is disproportionate to its value, same call made for v2)
with open(os.path.join(MODELS_DIR, "best_model_v3.pkl"), "wb") as f:
    pickle.dump(best_model_v3, f)
size_mb = os.path.getsize(os.path.join(MODELS_DIR, "best_model_v3.pkl")) / 1e6
print(f"best_model_v3.pkl size: {size_mb:.2f} MB")

model_metrics_v3 = {k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in results.items()}
model_metrics_v3["cv_best"] = {"mean_r2": cv_scores_v3.mean(), "std_r2": cv_scores_v3.std(), "model": best_name}
model_metrics_v3["cv_v2_baseline"] = {"mean_r2": cv_scores_v2.mean(), "std_r2": cv_scores_v2.std()}
with open(os.path.join(MODELS_DIR, "model_metrics_v3.pkl"), "wb") as f:
    pickle.dump(model_metrics_v3, f)

importances = pd.Series(best_model_v3.feature_importances_, index=X_v3.columns).sort_values(ascending=False)
print("\nTop 15 features:")
for feat, imp in importances.head(15).items():
    print(f"  {feat:30s}: {imp:.4f}")

print(f"\nTotal time: {time.time()-t0:.1f}s")


best_model_v3.pkl size: 1.34 MB

Top 15 features:
  aryl_fp_46                    : 0.2791
  aryl_fp_49                    : 0.1050
  product_mw                    : 0.0917
  additive_ethyl-isoxazole-4-carboxylate: 0.0819
  aryl_fp_65                    : 0.0379
  additive_5-Phenyl-1,2,4-oxadiazole: 0.0351
  additive_benzo_c_isoxazole    : 0.0349
  additive_ethyl-5-methylisoxazole-4-carboxylate: 0.0280
  additive_methyl-isoxazole-5-carboxylate: 0.0231
  aryl_fp_5                     : 0.0199
  aryl_fp_88                    : 0.0187
  ligand_XPhos                  : 0.0142
  aryl_fp_101                   : 0.0131
  base_MTBD                     : 0.0116
  additive_ethyl-3-methylisoxazole-5-carboxylate: 0.0112

Total time: 24.0s


## Result & next steps

**Test R²: 0.30 (v1, buggy) → 0.72 (v2, bug fixed) → 0.93 (v3, + Morgan fingerprints)**,
the last confirmed with 5-fold CV (mean 0.933, std 0.005) rather than a single split.

Notably, the top-2 most important features are now fingerprint bits (`aryl_fp_46`, `aryl_fp_49`) —
i.e. specific substructure fragments of the aryl halide — ranking above every individual physicochemical
descriptor. This is a sensible result: circular fingerprints let the model key directly on the presence/
absence of structural motifs, which bulk descriptors like MW or LogP cannot represent.

`app.py` has been updated to use this v3 model (XGBoost, `best_model_v3.pkl`) for its live predictions:
it now computes descriptors *and* fingerprints from the user's actual SMILES input.

**Possible further steps (not pursued here):**
- Hyperparameter tuning (GridSearch/RandomizedSearch) on the v3 feature set
- Larger fingerprint bit length (256/512) or including fingerprints on the amine nucleophile too
- MACCS keys as an additional/alternative fingerprint type
